In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Considere el proyecto cuyo cuadro de prelaciones y duraciones (en **días laborables**) se adjunta. El proyecto empieza el lunes 2026-09-28. Se trabaja de lunes a viernes; son festivos los días 2026-10-09 y 2026-11-03.

In [4]:
pregunta_1 = "1. Calcule el calendario del proyecto en días laborables (inicio y fin más tempranos y más tardíos, holgura total) y su duración.  "
pregunta_2 = "2. Traduzca el calendario a fechas: fecha de inicio y de fin (último día trabajado) más tempranas y más tardías de cada actividad. ¿En qué fecha termina el proyecto? ¿Cuántos días naturales dura?  "
pregunta_3 = "3. ¿Qué actividades críticas atraviesan un festivo o un fin de semana? ¿Cambia eso su duración en días laborables? ¿Y la fecha de fin del proyecto?  "
pregunta_4 = "4. Se declara un nuevo festivo el 2026-10-02. ¿Qué fechas cambian y cuánto se retrasa la entrega? ¿Depende de que el festivo caiga en una actividad crítica?  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Calcule el calendario del proyecto en días laborables (inicio y fin más tempranos y más tardíos, holgura total) y su duración.  

2. Traduzca el calendario a fechas: fecha de inicio y de fin (último día trabajado) más tempranas y más tardías de cada actividad. ¿En qué fecha termina el proyecto? ¿Cuántos días naturales dura?  

3. ¿Qué actividades críticas atraviesan un festivo o un fin de semana? ¿Cambia eso su duración en días laborables? ¿Y la fecha de fin del proyecto?  

4. Se declara un nuevo festivo el 2026-10-02. ¿Qué fechas cambian y cuánto se retrasa la entrega? ¿Depende de que el festivo caiga en una actividad crítica?  

In [5]:
#| tbl-cap: "Cuadro de datos del enunciado"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration
    ('A'     , '---'      , 8       ),
    ('B'     , '---'      , 5       ),
    ('C'     , '---'      , 1       ),
    ('D'     , 'B,C'      , 4       ),
    ('E'     , 'A,D'      , 3       ),
    ('F'     , 'C'        , 5       ),
    ('G'     , 'D'        , 2       ),
    ('H'     , 'E,G'      , 7       ),
    ('I'     , 'F,H'      , 3       ),
], columns=['activity', 'predecessor', 'duration']).set_index('activity')
datos_enunciado

,predecessor,duration
activity,,
A,---,8
B,---,5
C,---,1
D,"B,C",4
E,"A,D",3
F,C,5
G,D,2
H,"E,G",7
I,"F,H",3


El proyecto corresponde a la instalación de una planta fotovoltaica sobre cubierta; las actividades son las siguientes (las duraciones del cuadro están en días).

In [6]:
#| tbl-cap: "Descripción de las actividades"
#| label: tbl-descripciones
descripciones = pd.Series({
    'A': 'Estudio de sombras y producción',
    'B': 'Cálculo estructural de la cubierta',
    'C': 'Tramitación de permisos',
    'D': 'Acopio de módulos e inversores',
    'E': 'Montaje de la estructura soporte',
    'F': 'Colocación de módulos',
    'G': 'Cableado de corriente continua',
    'H': 'Instalación de inversores',
    'I': 'Cuadro de protecciones y contador',
}, name='Descripción')
descripciones.to_frame()

,Descripción
A,Estudio de sombras y producción
B,Cálculo estructural de la cubierta
C,Tramitación de permisos
D,Acopio de módulos e inversores
E,Montaje de la estructura soporte
F,Colocación de módulos
G,Cableado de corriente continua
H,Instalación de inversores
I,Cuadro de protecciones y contador


# Resolución

In [7]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
inicio = np.datetime64('2026-09-28')
festivos = [np.datetime64('2026-10-09'), np.datetime64('2026-11-03')]

## Apartado 1

In [8]:
display(Markdown(pregunta_1))

1. Calcule el calendario del proyecto en días laborables (inicio y fin más tempranos y más tardíos, holgura total) y su duración.  

In [9]:
#| tbl-cap: "Calendario en días laborables"
#| label: tbl-calendario
calendario = proyecto.calendar(datos_proyecto['duration'])
duracion = proyecto.duration(datos_proyecto['duration'])
proyecto.display_calendario(datos_proyecto['duration'])

,inicio_mas_temprano,fin_mas_temprano,inicio_mas_tardio,fin_mas_tardio,H_total
activity,,,,,
A,0,8,1,9,1
B,0,5,0,5,0
C,0,1,4,5,4
D,5,9,5,9,0
E,9,12,9,12,0
F,1,6,14,19,13
G,9,11,10,12,1
H,12,19,12,19,0
I,19,22,19,22,0


## Apartado 2

In [10]:
display(Markdown(pregunta_2))

2. Traduzca el calendario a fechas: fecha de inicio y de fin (último día trabajado) más tempranas y más tardías de cada actividad. ¿En qué fecha termina el proyecto? ¿Cuántos días naturales dura?  

El día laborable $k$ (contando desde 0) es `busday_offset(inicio, k)`, que salta fines de semana y festivos. Una actividad que empieza en el laborable ES y dura d ocupa los laborables ES, …, ES + d − 1: su último día trabajado es `busday_offset(inicio, EF − 1)`.

In [11]:
def fecha(k):
    return np.busday_offset(inicio, int(k), roll='forward', holidays=festivos)
fechas = pd.DataFrame({
    'inicio temprano': [fecha(k) for k in calendario['inicio_mas_temprano']],
    'fin temprano': [fecha(k - 1) for k in calendario['fin_mas_temprano']],
    'inicio tardío': [fecha(k) for k in calendario['inicio_mas_tardio']],
    'fin tardío': [fecha(k - 1) for k in calendario['fin_mas_tardio']]}, index=calendario.index)
fechas

,inicio temprano,fin temprano,inicio tardío,fin tardío
activity,,,,
A,2026-09-28,2026-10-07,2026-09-29,2026-10-08
B,2026-09-28,2026-10-02,2026-09-28,2026-10-02
C,2026-09-28,2026-09-28,2026-10-02,2026-10-02
D,2026-10-05,2026-10-08,2026-10-05,2026-10-08
E,2026-10-12,2026-10-14,2026-10-12,2026-10-14
F,2026-09-29,2026-10-05,2026-10-19,2026-10-23
G,2026-10-12,2026-10-13,2026-10-13,2026-10-14
H,2026-10-15,2026-10-23,2026-10-15,2026-10-23
I,2026-10-26,2026-10-28,2026-10-26,2026-10-28


In [12]:
DIAS = {'Monday': 'lunes', 'Tuesday': 'martes', 'Wednesday': 'miércoles', 'Thursday': 'jueves', 'Friday': 'viernes', 'Saturday': 'sábado', 'Sunday': 'domingo'}
fin_proyecto = fecha(duracion - 1)
dias_naturales = int((fin_proyecto - inicio).astype(int)) + 1
Markdown(f'El proyecto termina el {fin_proyecto} ({DIAS[pd.Timestamp(fin_proyecto).day_name()]}): {duracion} días laborables son {dias_naturales} días naturales, '
         f'{dias_naturales - duracion} de ellos no trabajados ({int(np.busday_count(inicio, fin_proyecto + np.timedelta64(1, "D"), weekmask="Sat Sun"))} de fin de semana y {len([f for f in festivos if inicio <= f <= fin_proyecto])} festivos).')

El proyecto termina el 2026-10-28 (miércoles): 22 días laborables son 31 días naturales, 9 de ellos no trabajados (8 de fin de semana y 1 festivos).

## Apartado 3

In [13]:
display(Markdown(pregunta_3))

3. ¿Qué actividades críticas atraviesan un festivo o un fin de semana? ¿Cambia eso su duración en días laborables? ¿Y la fecha de fin del proyecto?  

In [14]:
def dias_no_laborables(ini, fin):
    todos = np.arange(ini, fin + np.timedelta64(1, 'D'), dtype='datetime64[D]')
    return int(sum(1 for x in todos if not np.is_busday(x, holidays=festivos)))
criticas = calendario.index[calendario['H_total'] == 0]
cruce = pd.DataFrame({'inicio': fechas.loc[criticas, 'inicio temprano'], 'fin': fechas.loc[criticas, 'fin temprano'],
                      'días no laborables que atraviesa': [dias_no_laborables(fechas.loc[a, 'inicio temprano'], fechas.loc[a, 'fin temprano']) for a in criticas],
                      'incluye festivo': [any(fechas.loc[a, 'inicio temprano'] <= f <= fechas.loc[a, 'fin temprano'] for f in festivos) for a in criticas]})
display(cruce)
Markdown('La duración en días laborables no cambia: los festivos y fines de semana sólo estiran la actividad en el calendario natural. '
         'La fecha de fin del proyecto sí depende de cuántos días no laborables caen dentro del horizonte, estén donde estén.')

,inicio,fin,días no laborables que atraviesa,incluye festivo
activity,,,,
B,2026-09-28,2026-10-02,0,False
D,2026-10-05,2026-10-08,0,False
E,2026-10-12,2026-10-14,0,False
H,2026-10-15,2026-10-23,2,False
I,2026-10-26,2026-10-28,0,False


La duración en días laborables no cambia: los festivos y fines de semana sólo estiran la actividad en el calendario natural. La fecha de fin del proyecto sí depende de cuántos días no laborables caen dentro del horizonte, estén donde estén.

## Apartado 4

In [15]:
display(Markdown(pregunta_4))

4. Se declara un nuevo festivo el 2026-10-02. ¿Qué fechas cambian y cuánto se retrasa la entrega? ¿Depende de que el festivo caiga en una actividad crítica?  

In [16]:
festivo_extra = np.datetime64('2026-10-02')
festivos2 = sorted(festivos + [festivo_extra])
def fecha2(k):
    return np.busday_offset(inicio, int(k), roll='forward', holidays=festivos2)
fechas2 = pd.DataFrame({'fin temprano (antes)': fechas['fin temprano'], 'fin temprano (con el nuevo festivo)': [fecha2(k - 1) for k in calendario['fin_mas_temprano']]})
fechas2['cambia'] = fechas2.iloc[:, 0] != fechas2.iloc[:, 1]
display(fechas2)
fin2 = fecha2(duracion - 1)
Markdown(f'''El nuevo festivo desplaza un día natural todas las fechas posteriores a él ({", ".join(fechas2.index[fechas2["cambia"]])}), críticas o no: la entrega pasa del {fin_proyecto} al {fin2}. No depende de la criticidad, porque el festivo no consume días laborables de ninguna actividad: desplaza el calendario entero. Sólo si el festivo cayera después del fin del proyecto no afectaría.''')

,fin temprano (antes),fin temprano (con el nuevo festivo),cambia
activity,,,
A,2026-10-07,2026-10-08,True
B,2026-10-02,2026-10-05,True
C,2026-09-28,2026-09-28,False
D,2026-10-08,2026-10-12,True
E,2026-10-14,2026-10-15,True
F,2026-10-05,2026-10-06,True
G,2026-10-13,2026-10-14,True
H,2026-10-23,2026-10-26,True
I,2026-10-28,2026-10-29,True


El nuevo festivo desplaza un día natural todas las fechas posteriores a él (A, B, D, E, F, G, H, I), críticas o no: la entrega pasa del 2026-10-28 al 2026-10-29. No depende de la criticidad, porque el festivo no consume días laborables de ninguna actividad: desplaza el calendario entero. Sólo si el festivo cayera después del fin del proyecto no afectaría.